In [ ]:
# =============================
# 🔹 AdvancedResidualAutoencoder Ablation Study
# =============================
import os
import math
import h5py
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
import random
import warnings
import time
from scipy import stats

warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Create results directory if it doesn't exist
results_dir = "ablation_study_results"
os.makedirs(results_dir, exist_ok=True)
print(f"Results will be saved in: {os.path.abspath(results_dir)}")

print("Starting AdvancedResidualAutoencoder Ablation Study...")

# =============================
# 🔹 Load and Filter Metadata
# =============================
csv_file = r"C:\\Users\\hafiz\\Desktop\\Mostafa seismic signals\\seismic data\\merged.csv"
df = pd.read_csv(csv_file, low_memory=False)
df = df[(df.trace_category == 'earthquake_local') &
        (df.source_distance_km <= 60) &
        (df.source_magnitude > 3)]
trace_names = df['trace_name'].to_list()

print(f"Found {len(trace_names)} seismic traces")


# =============================
# 🔹 Efficient Data Loading
# =============================
def load_filtered_z_waveforms(hdf5_file, trace_names, group='data', min_len=1500, max_samples=None):
    """Load waveforms efficiently without normalization"""
    data = []
    if max_samples is None:
        max_samples = len(trace_names)

    print(f"Loading up to {max_samples} waveforms...")

    with h5py.File(hdf5_file, 'r') as f:
        for i, name in enumerate(trace_names):
            if i >= max_samples:
                break
            try:
                waveform = f[group][name][:]
                if waveform.shape[0] >= min_len:
                    data.append(waveform[:min_len, 0])
            except KeyError:
                continue

            if (i + 1) % 1000 == 0:
                print(f"Processed {i + 1}/{min(max_samples, len(trace_names))} waveforms")

    print(f"Successfully loaded {len(data)} waveforms")
    return np.array(data)


file_name = r"C:\\Users\\hafiz\\Desktop\\Mostafa seismic signals\\seismic data\\merged.hdf5"
waveforms = load_filtered_z_waveforms(file_name, trace_names, max_samples=None)
print(f"Raw waveforms shape: {waveforms.shape}")


# =============================
# 🔹 Memory-Efficient Dataset
# =============================
class EfficientSeismicDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        waveform = self.data[idx]
        z_min = np.min(waveform)
        z_max = np.max(waveform)

        if z_max - z_min > 0:
            waveform = (waveform - z_min) / (z_max - z_min)
        else:
            waveform = np.zeros_like(waveform)

        return torch.tensor(waveform, dtype=torch.float32).unsqueeze(0)


# =============================
# 🔹 ABLATION MODEL VARIANTS
# =============================

class AttentionBlock(nn.Module):
    """Simplified channel attention mechanism"""

    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, max(4, channels // 8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(max(4, channels // 8), channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights


class EnhancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.elu(out)
        out = self.attention(out)
        return out


class PlainConvBlock(nn.Module):
    """Plain convolutional block without residual connections"""

    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.elu(out)
        out = self.attention(out)
        return out


# =============================
# 🔹 ABLATION 1: BASE MODEL (Your Original)
# =============================
class AdvancedResidualAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        self.target_length = max(16, 1500 // cr)

        # Build encoder based on compression ratio
        encoder_layers = [
            # Initial convolution
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ELU(inplace=True),
            AttentionBlock(32),

            # Residual blocks with attention
            EnhancedResidualBlock(32, 64, stride=2, use_attention=True),
            EnhancedResidualBlock(64, 128, stride=2, use_attention=True),
        ]

        # ARCHITECTURE TWEAK FOR HIGH CR: Add extra layers for CR > 30
        if cr > 30:
            encoder_layers.append(EnhancedResidualBlock(128, 256, stride=2, use_attention=True))
            final_channels = 256
        else:
            final_channels = 128

        # Final compression
        encoder_layers.append(nn.AdaptiveAvgPool1d(self.target_length))
        self.encoder = nn.Sequential(*encoder_layers)

        # Calculate upsampling factors
        self.upsample_factors = self.calculate_upsample_factors()

        # Build decoder
        decoder_layers = []
        current_channels = final_channels

        for i, factor in enumerate(self.upsample_factors):
            next_channels = max(16, current_channels // 2)

            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                EnhancedResidualBlock(current_channels, next_channels, stride=1, use_attention=True)
            ])
            current_channels = next_channels

        # Final layers
        decoder_layers.extend([
            nn.Conv1d(current_channels, 1, kernel_size=15, padding=7),
            nn.Sigmoid()
        ])

        self.decoder = nn.Sequential(*decoder_layers)

    def calculate_upsample_factors(self):
        """Calculate exact upsampling factors safely"""
        current_length = self.target_length
        factors = []

        while current_length < 1500:
            remaining_ratio = 1500 / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)

            if len(factors) >= 5:
                break

        if current_length != 1500:
            final_factor = 1500 / current_length
            factors.append(final_factor)

        return factors

    def forward(self, x):
        encoded = self.encoder(x)
        return self.decoder(encoded)


# =============================
# 🔹 ABLATION 2: NO ATTENTION
# =============================
class NoAttentionAdvancedResidualAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        self.target_length = max(16, 1500 // cr)

        # Build encoder WITHOUT attention blocks
        encoder_layers = [
            # Initial convolution
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ELU(inplace=True),
            # NO AttentionBlock here

            # Residual blocks WITHOUT attention
            EnhancedResidualBlock(32, 64, stride=2, use_attention=False),
            EnhancedResidualBlock(64, 128, stride=2, use_attention=False),
        ]

        if cr > 30:
            encoder_layers.append(EnhancedResidualBlock(128, 256, stride=2, use_attention=False))
            final_channels = 256
        else:
            final_channels = 128

        encoder_layers.append(nn.AdaptiveAvgPool1d(self.target_length))
        self.encoder = nn.Sequential(*encoder_layers)

        # Calculate upsampling factors
        self.upsample_factors = self.calculate_upsample_factors()

        # Build decoder WITHOUT attention
        decoder_layers = []
        current_channels = final_channels

        for i, factor in enumerate(self.upsample_factors):
            next_channels = max(16, current_channels // 2)

            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                EnhancedResidualBlock(current_channels, next_channels, stride=1, use_attention=False)  # No attention
            ])
            current_channels = next_channels

        decoder_layers.extend([
            nn.Conv1d(current_channels, 1, kernel_size=15, padding=7),
            nn.Sigmoid()
        ])

        self.decoder = nn.Sequential(*decoder_layers)

    def calculate_upsample_factors(self):
        """Calculate exact upsampling factors safely"""
        current_length = self.target_length
        factors = []

        while current_length < 1500:
            remaining_ratio = 1500 / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)

            if len(factors) >= 5:
                break

        if current_length != 1500:
            final_factor = 1500 / current_length
            factors.append(final_factor)

        return factors

    def forward(self, x):
        encoded = self.encoder(x)
        return self.decoder(encoded)


# =============================
# 🔹 ABLATION 3: PLAIN CONVOLUTION (No Residual)
# =============================
class PlainConvAdvancedAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        self.target_length = max(16, 1500 // cr)

        # Build encoder with PLAIN convolution blocks (no residual)
        encoder_layers = [
            # Initial convolution
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ELU(inplace=True),
            AttentionBlock(32),

            # Plain convolution blocks instead of residual
            PlainConvBlock(32, 64, stride=2, use_attention=True),
            PlainConvBlock(64, 128, stride=2, use_attention=True),
        ]

        if cr > 30:
            encoder_layers.append(PlainConvBlock(128, 256, stride=2, use_attention=True))
            final_channels = 256
        else:
            final_channels = 128

        encoder_layers.append(nn.AdaptiveAvgPool1d(self.target_length))
        self.encoder = nn.Sequential(*encoder_layers)

        # Calculate upsampling factors
        self.upsample_factors = self.calculate_upsample_factors()

        # Build decoder with plain convolution
        decoder_layers = []
        current_channels = final_channels

        for i, factor in enumerate(self.upsample_factors):
            next_channels = max(16, current_channels // 2)

            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                PlainConvBlock(current_channels, next_channels, stride=1, use_attention=True)
            ])
            current_channels = next_channels

        decoder_layers.extend([
            nn.Conv1d(current_channels, 1, kernel_size=15, padding=7),
            nn.Sigmoid()
        ])

        self.decoder = nn.Sequential(*decoder_layers)

    def calculate_upsample_factors(self):
        """Calculate exact upsampling factors safely"""
        current_length = self.target_length
        factors = []

        while current_length < 1500:
            remaining_ratio = 1500 / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)

            if len(factors) >= 5:
                break

        if current_length != 1500:
            final_factor = 1500 / current_length
            factors.append(final_factor)

        return factors

    def forward(self, x):
        encoded = self.encoder(x)
        return self.decoder(encoded)


# =============================
# 🔹 ABLATION 4: MODEL CAPACITY VARIANTS
# =============================

# Light variant (50% reduction in channels)
class LightAdvancedResidualAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        self.target_length = max(16, 1500 // cr)

        # Reduced channel dimensions (50% of original)
        encoder_layers = [
            nn.Conv1d(1, 16, kernel_size=15, stride=2, padding=7),  # 32→16
            nn.BatchNorm1d(16),
            nn.ELU(inplace=True),
            AttentionBlock(16),

            EnhancedResidualBlock(16, 32, stride=2, use_attention=True),  # 64→32
            EnhancedResidualBlock(32, 64, stride=2, use_attention=True),  # 128→64
        ]

        if cr > 30:
            encoder_layers.append(EnhancedResidualBlock(64, 128, stride=2, use_attention=True))  # 256→128
            final_channels = 128
        else:
            final_channels = 64

        encoder_layers.append(nn.AdaptiveAvgPool1d(self.target_length))
        self.encoder = nn.Sequential(*encoder_layers)

        self.upsample_factors = self.calculate_upsample_factors()

        decoder_layers = []
        current_channels = final_channels

        for i, factor in enumerate(self.upsample_factors):
            next_channels = max(8, current_channels // 2)  # Reduced minimum

            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                EnhancedResidualBlock(current_channels, next_channels, stride=1, use_attention=True)
            ])
            current_channels = next_channels

        decoder_layers.extend([
            nn.Conv1d(current_channels, 1, kernel_size=15, padding=7),
            nn.Sigmoid()
        ])

        self.decoder = nn.Sequential(*decoder_layers)

    def calculate_upsample_factors(self):
        current_length = self.target_length
        factors = []

        while current_length < 1500:
            remaining_ratio = 1500 / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)

            if len(factors) >= 5:
                break

        if current_length != 1500:
            final_factor = 1500 / current_length
            factors.append(final_factor)

        return factors

    def forward(self, x):
        encoded = self.encoder(x)
        return self.decoder(encoded)


# Heavy variant (2x increase in channels)
class HeavyAdvancedResidualAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        self.target_length = max(16, 1500 // cr)

        # Increased channel dimensions (2x original)
        encoder_layers = [
            nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7),  # 32→64
            nn.BatchNorm1d(64),
            nn.ELU(inplace=True),
            AttentionBlock(64),

            EnhancedResidualBlock(64, 128, stride=2, use_attention=True),  # 64→128
            EnhancedResidualBlock(128, 256, stride=2, use_attention=True),  # 128→256
        ]

        if cr > 30:
            encoder_layers.append(EnhancedResidualBlock(256, 512, stride=2, use_attention=True))  # 256→512
            final_channels = 512
        else:
            final_channels = 256

        encoder_layers.append(nn.AdaptiveAvgPool1d(self.target_length))
        self.encoder = nn.Sequential(*encoder_layers)

        self.upsample_factors = self.calculate_upsample_factors()

        decoder_layers = []
        current_channels = final_channels

        for i, factor in enumerate(self.upsample_factors):
            next_channels = max(32, current_channels // 2)  # Increased minimum

            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                EnhancedResidualBlock(current_channels, next_channels, stride=1, use_attention=True)
            ])
            current_channels = next_channels

        decoder_layers.extend([
            nn.Conv1d(current_channels, 1, kernel_size=15, padding=7),
            nn.Sigmoid()
        ])

        self.decoder = nn.Sequential(*decoder_layers)

    def calculate_upsample_factors(self):
        current_length = self.target_length
        factors = []

        while current_length < 1500:
            remaining_ratio = 1500 / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)

            if len(factors) >= 5:
                break

        if current_length != 1500:
            final_factor = 1500 / current_length
            factors.append(final_factor)

        return factors

    def forward(self, x):
        encoded = self.encoder(x)
        return self.decoder(encoded)


# =============================
# 🔹 UPDATED Training Function
# =============================
def train_model(model, train_loader, val_loader, model_name, cr, max_epochs=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training {model_name} CR={cr} on {device}")

    # IMPROVED Progressive epoch assignment
    if max_epochs is None:
        if cr <= 5:
            max_epochs = 120  # Increased for low CR
        elif cr <= 15:
            max_epochs = 100
        elif cr <= 30:
            max_epochs = 100
        else:
            max_epochs = 100

    # IMPROVED Learning rates for better convergence
    if cr <= 3:  # Special handling for very low CR
        lr = 0.0002  # Lower LR for stability
    elif cr <= 5:
        lr = 0.0002
    elif cr <= 15:
        lr = 0.0002
    else:
        lr = 0.0002

    print(f"Using {max_epochs} epochs, LR={lr} for CR={cr}")

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)  # Added regularization

    # IMPROVED scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True, min_lr=1e-6
    )

    # EARLY STOPPING
    best_val_loss = float('inf')
    patience = 8
    patience_counter = 0
    best_model_state = None

    train_losses = []
    val_losses = []

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            inputs = batch.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = nn.MSELoss()(inputs, outputs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch.to(device)
                outputs = model(inputs)
                loss = nn.MSELoss()(inputs, outputs)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")

        # EARLY STOPPING CHECK
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

        if epoch % 2 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, train_losses, val_losses


# =============================
# 🔹 Evaluation Metrics
# =============================
def calculate_psnr(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr


def calculate_ssim_1d(original, reconstructed, data_range=1.0):
    if len(original.shape) > 1:
        original = original.flatten()
    if len(reconstructed.shape) > 1:
        reconstructed = reconstructed.flatten()
    return ssim(original, reconstructed, data_range=data_range)


def calculate_model_size(model):
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    size_mb = (param_size + buffer_size) / 1024 ** 2
    return size_mb


def evaluate_model(model, test_data, cr, model_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    with torch.no_grad():
        test_samples = test_data[:10]  # Use first 10 samples for evaluation
        inputs = torch.tensor(test_samples, dtype=torch.float32).to(device)
        recon = model(inputs).cpu().numpy()
        x = inputs.cpu().numpy()

    # Calculate metrics
    orig_flat = x[:, 0, :].flatten()
    recon_flat = recon[:, 0, :].flatten()

    mse = np.mean((orig_flat - recon_flat) ** 2)
    signal_power = np.mean(orig_flat ** 2)
    snr = 10 * math.log10(signal_power / mse) if mse > 0 else float('inf')

    if np.std(orig_flat) > 0 and np.std(recon_flat) > 0:
        correlation = np.corrcoef(orig_flat, recon_flat)[0, 1]
    else:
        correlation = 0

    mae = np.mean(np.abs(orig_flat - recon_flat))
    psnr = calculate_psnr(orig_flat, recon_flat)
    ssim_value = calculate_ssim_1d(orig_flat, recon_flat)

    # Calculate actual compression ratio
    test_input = torch.randn(1, 1, 1500).to(device)
    compressed = model.encoder(test_input)
    actual_cr = 1500 / compressed.shape[-1]
    cr_error = abs(actual_cr - cr) / cr * 100

    # Calculate model complexity
    model_size = calculate_model_size(model)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())

    return {
        "MSE": mse,
        "SNR": snr,
        "Correlation": correlation,
        "MAE": mae,
        "PSNR": psnr,
        "SSIM": ssim_value,
        "CR_Error": cr_error,
        "Model_Size_MB": model_size,
        "Parameter_Count": total_params,
        "Model": model_name,
        "CR": cr,
        "Actual_CR": actual_cr,
        "Reconstructions": (x[:, 0, :], recon[:, 0, :])
    }


# =============================
# 🔹 Visualization Functions
# =============================
def plot_waveforms(original, reconstructed, cr, model_name, save_figures=True):
    fig, axes = plt.subplots(min(3, len(original)), 2, figsize=(12, 2.5 * min(3, len(original))))

    if min(3, len(original)) == 1:
        axes = np.array([axes])

    for i in range(min(3, len(original))):
        # Time domain
        axes[i, 0].plot(original[i], 'b-', label='Original', linewidth=1, alpha=0.8)
        axes[i, 0].plot(reconstructed[i], 'r-', label='Reconstructed', linewidth=1, alpha=0.8)
        axes[i, 0].set_title(f'{model_name} - CR={cr} - Sample {i + 1}')
        axes[i, 0].legend()
        axes[i, 0].grid(True, alpha=0.3)

        # Frequency domain
        orig_fft = np.abs(np.fft.fft(original[i]))
        recon_fft = np.abs(np.fft.fft(reconstructed[i]))
        freq = np.fft.fftfreq(len(original[i]))

        axes[i, 1].semilogy(freq[:len(freq) // 2], orig_fft[:len(freq) // 2], 'b-', label='Original')
        axes[i, 1].semilogy(freq[:len(freq) // 2], recon_fft[:len(freq) // 2], 'r-', label='Reconstructed')
        axes[i, 1].set_title('Frequency Spectrum')
        axes[i, 1].legend()
        axes[i, 1].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_figures:
        model_dir = os.path.join(results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)

        filename_base = f"{model_name}_CR{cr}_waveforms"
        png_path = os.path.join(model_dir, f"{filename_base}.png")

        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        print(f"Saved waveforms to {png_path}")

    plt.show()
    plt.close()


def plot_metrics_comparison(results_df, save_figures=True):
    models = results_df['Model'].unique()
    metrics = ['SNR', 'Correlation', 'PSNR', 'SSIM', 'MSE', 'MAE']

    # Create combined plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        for model in models:
            model_data = results_df[results_df['Model'] == model]
            if metric in ['MSE', 'MAE']:
                axes[i].loglog(model_data['CR'], model_data[metric], 'o-', label=model, markersize=6)
            else:
                axes[i].semilogx(model_data['CR'], model_data[metric], 'o-', label=model, markersize=6)

        axes[i].set_title(f'{metric} vs Compression Ratio')
        axes[i].set_xlabel('Compression Ratio')
        axes[i].set_ylabel(metric)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend()

    plt.tight_layout()

    if save_figures:
        # Save combined plot
        png_path = os.path.join(results_dir, "metrics_comparison.png")
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        print(f"Saved combined metrics comparison to {png_path}")

    plt.show()
    plt.close()

    # Create individual metric plots
    if save_figures:
        for metric in metrics:
            plt.figure(figsize=(10, 6))

            for model in models:
                model_data = results_df[results_df['Model'] == model]
                if metric in ['MSE', 'MAE']:
                    plt.loglog(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)
                else:
                    plt.semilogx(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)

            plt.title(f'{metric} vs Compression Ratio', fontsize=14, fontweight='bold')
            plt.xlabel('Compression Ratio', fontsize=12)
            plt.ylabel(metric, fontsize=12)
            plt.grid(True, alpha=0.3)
            plt.legend(fontsize=10)
            plt.tight_layout()

            # Save individual metric plot
            individual_path = os.path.join(results_dir, f"{metric.lower()}_comparison.png")
            plt.savefig(individual_path, dpi=300, bbox_inches='tight')
            print(f"Saved {metric} comparison to {individual_path}")
            plt.close()


# =============================
# 🔹 Main Ablation Study
# =============================
def main():
    # Create dataset
    dataset = EfficientSeismicDataset(waveforms)
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = train_test_split(dataset, train_size=train_size, test_size=val_size, random_state=42)

    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    # Test data
    test_data = np.array([val_dataset[i] for i in range(min(10, len(val_dataset)))])  # Smaller test set

    print(f"Training on {len(train_dataset)} samples, testing on {len(test_data)} samples")

    # Compression ratios to test
    compression_ratios = [2, 3, 5, 10, 20, 50, 100]  # Updated with low CR values

    # Define ablation models to compare
    ablation_models = [
        ("Base_Residual_Attention", AdvancedResidualAutoencoder),  # Your original
        ("No_Attention", NoAttentionAdvancedResidualAutoencoder),  # Ablation 1
        ("Plain_Conv", PlainConvAdvancedAutoencoder),  # Ablation 2
        ("Light_Capacity", LightAdvancedResidualAutoencoder),  # Ablation 3
        ("Heavy_Capacity", HeavyAdvancedResidualAutoencoder),  # Ablation 3
    ]

    results = []
    training_history = {}

    print("Starting AdvancedResidualAutoencoder Ablation Study...")
    print("=" * 80)

    for model_name, model_class in ablation_models:
        print(f"\n=== Ablation: {model_name} ===")
        model_cr_results = []

        for cr in compression_ratios:
            print(f"\nTraining {model_name} CR={cr}")
            print("-" * 40)

            try:
                # Initialize and train model
                model = model_class(cr)
                trained_model, train_losses, val_losses = train_model(
                    model, train_loader, val_loader, model_name, cr
                )

                # Store training history
                training_history[f"{model_name}_CR{cr}"] = (train_losses, val_losses)

                # Evaluate model
                metrics = evaluate_model(trained_model, test_data, cr, model_name)
                results.append(metrics)
                model_cr_results.append(metrics)

                # Plot sample reconstructions
                orig, recon = metrics["Reconstructions"]
                plot_waveforms(orig, recon, cr, model_name, save_figures=True)

                print(f"{model_name} CR={cr}: "
                      f"SNR={metrics['SNR']:.2f}dB, PSNR={metrics['PSNR']:.2f}dB, "
                      f"SSIM={metrics['SSIM']:.4f}, Params={metrics['Parameter_Count']:,}")

            except Exception as e:
                print(f"Error training {model_name} CR={cr}: {e}")
                continue

    # Save and analyze results
    if results:
        results_df = pd.DataFrame(results)
        csv_path = os.path.join(results_dir, 'ablation_study_results.csv')
        results_df.to_csv(csv_path, index=False)
        print(f"\nResults saved to {csv_path}")

        print("\nAblation Study Results Summary:")
        print("=" * 120)
        print(results_df.round(4))

        # Plot comprehensive comparisons
        plot_metrics_comparison(results_df, save_figures=True)

        # Statistical analysis
        print("\nStatistical Analysis (SNR across all CRs):")
        models = results_df['Model'].unique()
        for model in models:
            model_snr = results_df[results_df['Model'] == model]['SNR']
            print(f"{model}: SNR = {model_snr.mean():.2f} ± {model_snr.std():.2f} dB")

        # Parameter efficiency analysis
        print("\nParameter Efficiency (SNR per Million Parameters):")
        for model in models:
            model_data = results_df[results_df['Model'] == model]
            avg_snr = model_data['SNR'].mean()
            avg_params = model_data['Parameter_Count'].mean() / 1e6  # Millions
            efficiency = avg_snr / avg_params if avg_params > 0 else 0
            print(f"{model}: {efficiency:.2f} dB per million parameters")

    print("\nAblation study completed!")
    print(f"All results saved in: {os.path.abspath(results_dir)}")


if __name__ == "__main__":
    main()
